In [106]:
#Base Do CRUD Organizada Em Funções Com Tratamento Básico De Erros.


In [107]:
import psycopg
conexao = psycopg.connect(
    dbname = "bancodedados",
    user = "postgres",
    password = "22082008jp",
    host = "localhost",
    port = 5432
)
cursor = conexao.cursor()
conexao.rollback()

In [108]:
def Cadastro():
    while True:
        nome = input("Digite O Nome Do Produto:")
        if not nome:
            print("ERRO: É Necessário Digitar Um Nome!")
            continue
        nome = nome.lower()
        resultado = cursor.execute("SELECT NomeProduto FROM Produtos WHERE NomeProduto = %s", (nome,))
        item = resultado.fetchone()
        if item is not None:
            print("ERRO: Produto Já Cadastrado!")
            return None
        else:
            print("SISTEMA: Nome Validado!")
            break
    while True:
        nome_categoria = input("Digite A Categoria Do Produto:")
        if not nome_categoria:
            print("ERRO: É Necessário Digitar Uma Categoria!")
            continue
        nome_categoria = nome_categoria.lower()
        resultado = cursor.execute("SELECT categoriapk FROM Categorias WHERE nomecategoria = %s",(nome_categoria,))
        chave = resultado.fetchone()
        if chave is None:
            print("ERRO: Categoria Não Encontrada!")
            return None
        else:
            valor_chave = chave[0]
            print("SISTEMA: Categoria Encontrada!")
            break 
    while True:
        try:
            preco = float(input("Digite O Preço:"))
            if not preco:
                print("ERRO: É Necessário Digitar Um Preço!")
                continue
            if preco < 0:
                print("ERRO: Preço Não Pode Ser Menor Do Que Zero!")
                continue
            if preco == 0:
                print("ERRO: Preço Não Pode Ser Zero!")
                continue
            print("SISTEMA: Preço Validado!")
            break
        except ValueError:
            print("ERRO: Digite Um Caractere Válido!")
    while True:
        try:
            quantidade = int(input("Digite A Quantidade Inicial Do Produto:"))
            if not quantidade:
                print("ERRO: É Necessário Digitar Uma Quantidade!")
                continue
            if quantidade < 0:
                print("ERRO: Quantidade Não Pode Ser Menor Que Zero!")
                continue
            if quantidade == 0:
                print("ERRO: Quantidade Inicial Não Pode Ser Zero!")
                continue
            print("SISTEMA: Quantidade Validada!")
            break
        except ValueError:
            print("ERRO: Digite Um Caractere Válido!")
    cursor.execute("""INSERT INTO produtos (nomeproduto, categoriafk, preco, quantidade)
                        VALUES (%s,%s,%s,%s)""",(nome,valor_chave,preco,quantidade))
    conexao.commit()

In [109]:
def Cadastro_Categoria():
    while True:
        nome_categoria = input("Digite O Nome De Registro Da Categoria:")
        if not nome_categoria:
            print("ERRO: É Necessário Digitar Um Nome!")
            continue
        nome_categoria = nome_categoria.lower()
        resultado = cursor.execute("SELECT CategoriaPK FROM Categorias WHERE NomeCategoria = %s",(nome_categoria,))
        nome = resultado.fetchone()
        if nome is not None:
            print("ERRO: Categoria Já Existe!")
            return None
        else:
            cursor.execute("INSERT INTO Categorias (NomeCategoria) VALUES (%s)",(nome_categoria,))
            conexao.commit()
            print("SISTEMA: Categoria Cadastrada!")
            break

In [110]:
def Listagem():
    produtos = cursor.execute("SELECT * FROM Produtos")
    resultado = produtos.fetchall()
    for item in resultado:
        id,nome,categoriaFK,preco,quantidade = item
        categoria = cursor.execute("""
    SELECT NomeCategoria
    FROM Categorias 
    WHERE CategoriaPK = %s""",(categoriaFK,))
    resultado_categoria = categoria.fetchone()
    for item in resultado_categoria:
        nome_categoria = item
    for item in resultado:
        id,nome,categoriaFK,preco,quantidade = item
        print("ID:{}\nNome: {}\nCategoria: {}\nChave Da Categoria: {}\nPreço: R${}\nQuantidade: {} Unidades\n".format(id,nome,nome_categoria,categoriaFK,preco,quantidade))


In [111]:
def Busca():
    print("Digite 1 Para Busca Por ID, 2 Para Busca Por Nome:")
    while True:
        try:
            opcao = int(input("Digite A Opção Desejada:"))
            if not opcao:
                print("ERRO: É Necessário Digitar Uma Opção!")
                continue
            if opcao not in [1,2]:
                print("ERRO: Digite Uma Opção Válida!")
                continue
            break
        except ValueError:
            print("ERRO: Digite Um Caractere Válido!")
    if opcao == 1:
        entrada = input("Digite O ID:")
        busca = cursor.execute("SELECT * FROM Produtos WHERE ProdutoPK = %s",(entrada,))
        resultado = busca.fetchall()
        for item in resultado:
            id,nome,categoriaFK,preco,quantidade = item
        categoria = cursor.execute("""
    SELECT NomeCategoria
    FROM Categorias 
    WHERE CategoriaPK = %s""",(categoriaFK,))
        resultado2 = categoria.fetchone()
        for item in resultado2:
            nome_categoria = item
        print("ID:{}\nNome: {}\nCategoria: {}\nChave Da Categoria: {}\nPreço: R${}\nQuantidade: {} Unidades\n".format(id,nome,nome_categoria,categoriaFK,preco,quantidade))
        return None
    elif opcao == 2:
        entrada = input("Digite O Nome:")
        busca = cursor.execute("SELECT * FROM Produtos WHERE NomeProduto = %s",(entrada,))
        resultado = busca.fetchall()
        for item in resultado:
            id,nome,categoriaFK,preco,quantidade = item
        categoria = cursor.execute("""
    SELECT NomeCategoria
    FROM Categorias 
    WHERE CategoriaPK = %s""",(categoriaFK,))
        resultado2 = categoria.fetchone()
        for item in resultado2:
            nome_categoria = item
        print("ID:{}\nNome: {}\nCategoria: {}\nChave Da Categoria: {}\nPreço: R${}\nQuantidade: {} Unidades\n".format(id,nome,nome_categoria,categoriaFK,preco,quantidade))
        return None

In [112]:
def Atualizar():
    pass

In [113]:
def Deletar():
    while True:
        try:
            busca = int(input("Digite O ID Do Produto:"))
            if not busca:
                print("ERRO: É Necessário Digitar Um ID!")
                continue
            if busca < 1: 
                print("ERRO: ID Sequencial, Valor Inicial a partir de 1.")
                continue
            break
        except:
            print("ERRO: Digite Um Caractere Válido!")
    select1 = cursor.execute("SELECT * FROM produtos WHERE produtopk = %s",(busca,))
    resultado = select1.fetchone()
    if resultado is None:
        print("ERRO: Produto Não Encontrado!")
        return None
    id,nome,categoriafk,preco,quantidade = resultado
    select2 = cursor.execute("SELECT nomecategoria FROM categorias WHERE categoriapk = %s",(categoriafk,))
    resultado = select2.fetchone()
    for categoria in resultado:
        print(categoria)
    print("Item Selecionado:")
    print("ID: {}\nNome: {}\n Categoria: {}\nChave Da Categoria: {}\nPreço: {}\nQuantidade: {}\n".format(id,nome,categoria,categoriafk,preco,quantidade))
    while True:
        confirmacao = input("Confirmar Deleção? (Y/N):")
        confirmacao = confirmacao.lower()
        if not confirmacao:
            print("ERRO: É Necessário Digitar Uma Opção!")
            continue
        if confirmacao not in ['y','n']:
            print("ERRO: Opção Inválida!")
            continue
        break
    if confirmacao == 'y':
        cursor.execute("DELETE FROM produtos WHERE produtopk = %s",(busca,))
        conexao.commit()
        print("Produto Deletado Com Sucesso!")

                    
            


In [114]:
def Entrada_E_Saida(dados):
    print("Opções:")
    print("1 - Entrada | 2 - Saída")
    while True:
        while True:
            opcao = int(input("Digite A Opção Desejada:"))
            if not opcao:
                print("Campo Não Pode Estar Vazio!")
                continue
            if opcao not in [1,2]:
                print("Opção Inválida!")
                continue
            break
        if opcao == 1:
            identificador = input("Digite O ID Do Produto A Atualizar:")
            if identificador in dados.keys():
                while True:
                    try:
                        quantidade = int(input("Digite A Quantidade:"))
                        if not quantidade:
                            print("Campo Não Pode Estar Vazio!")
                            continue
                        if quantidade < 0:
                            print("Quantidade Não Pode Ser Negativa!")
                            continue
                        break
                    except ValueError:
                        print("Digite Um Valor Válido!")
                for item,produto in dados.items():
                    print("Quantidade Alterada!")
                    produto.quantidade = produto.quantidade + quantidade
                    break
            break
        elif opcao == 2:
            identificador = input("Digite O ID Do Produto A Atualizar:")
            if identificador in dados.keys():
                while True:
                    try:
                        quantidade = int(input("Digite A Quantidade:"))
                        if not quantidade:
                            print("Campo Não Pode Estar Vazio!")
                            continue
                        if quantidade < 0:
                            print("Quantidade Não Pode Ser Negativa!")
                            continue
                        break
                    except ValueError:
                        print("Digite Um Valor Válido!")
                for item,produto in dados.items():
                    if quantidade > produto.quantidade:
                        print("Quantidade Insuficiente!")
                        break
                    else:
                        print("Quantidade Alterada!")
                        produto.quantidade = produto.quantidade - quantidade
                        break
                break
            else:
                print("Digite Uma Opção Válida!")

        
            

In [ ]:
print("Bem Vindo(a)! Abaixo Estão As Funções E Seus Respectivos Números")
print("----------------------------------------------------------------")
print("1 - Cadastrar Produto | 2 - Cadastrar Categoria | 3 - Listar Produtos ")
print("4 - Buscar Produto | 5 - Registrar Entrada Ou Saída ")



while True:
    while True:
        try:
            opcao = int(input("Digite A Função Desejada:"))
            break
        except ValueError:
            print("Digite Um Número Válido!")
            continue
    if opcao == 1:
        Cadastro()
    elif opcao == 2:
        Cadastro_Categoria()
    elif opcao == 3:
        Listagem()
    elif opcao == 4:
        Busca()
    elif opcao == 5:
       Deletar()
    elif opcao == 0:
        print("Saindo...")
        break
    

Bem Vindo(a)! Abaixo Estão As Funções E Seus Respectivos Números
----------------------------------------------------------------
1 - Cadastrar Produto | 2 - Cadastrar Categoria | 3 - Listar Produtos 
4 - Buscar Produto | 5 - Registrar Entrada Ou Saída 
bebida
Item Selecionado:
ID: 4
Nome: monster 
 Categoria: bebida
Chave Da Categoria: 2
Preço: 1.99
Quantidade: 10

Produto Deletado Com Sucesso!
fruta
Item Selecionado:
ID: 3
Nome: banana
 Categoria: fruta
Chave Da Categoria: 1
Preço: 20.99
Quantidade: 10

Produto Deletado Com Sucesso!
